# 03 – Data preparation, leakage control and validation (Tasks 4–8)

This notebook reads the outputs of `src/prepare_data.py`, `src/fit_transforms.py` and `src/validate_prepared.py` and re-queries the prepared views.

In [1]:
import json, sys
from pathlib import Path
import polars as pl
from IPython.display import Markdown, display
PKG = Path.cwd().parent
sys.path.insert(0, str(PKG / "src"))
META, PROCESSED = PKG / "data" / "metadata", PKG / "data" / "processed"
pl.Config.set_tbl_rows(30); pl.Config.set_tbl_cols(20); pl.Config.set_fmt_str_lengths(80)
show = lambda name: display(Markdown((META / name).read_text()))

## Preparation Decision Matrix (Task 4)

In [2]:
show('decision_matrix.md')

# Preparation Decision Matrix (Task 4, brief section 9)

Written before any data is changed. Each row names the issue (DQnn of `package/data/metadata/quality_issue_log.md`),
the decision, its rationale against the analytical questions, the implementation (function of
`package/src/prepare_data.py` unless named otherwise) and the check that validates it
(`package/src/validate_prepared.py`, check ID Vnn). Brief section 9.1 rules apply: no blanket imputation, no blanket
de-duplication, no IQR deletion, no arbitrary merging of categories, no future data in past features, sensitive
attributes kept for audit.

| Issue / need | Decision | Rationale | Implementation | Validation |
| :---- | :---- | :---- | :---- | :---- |
| Population (in-scope records) | Keep all 32,593 enrolments in the enrolment view with flags `enrolled_d13`, `enrolled_d14`, `enrolled_d27`; feature views hold only enrolments enrolled at their cut-off | The KPI counts students enrolled after the add-drop period (C03, C09); a list can only name students enrolled on the day it is drawn | `step2_population`: `enrolled_at(day)` = registered by day (or unknown) and not unregistered by day | V01 row counts 32,593 / 28,061 / 28,069 / 27,535 equal profile.json |
| Missing values (DQ04, DQ11–DQ14) | imd_band → category "Missing"; date_registration → null + `registration_missing`; withdrawal day of 93 Withdrawn → null + `withdrawal_day_missing`; TMA score → null; week_from/week_to not used | Missingness is informative (imd_band missing goes with 65.7% success, concentrated in two regions), small (45, 93, 173) or unused; imputing would invent values the data do not hold | `step5_missing`; filling of registration lead only in `fit_transforms.py`, median of training runs | V02 missing counts after = before for every kept column; V12 transform parameters computed from training rows only |
| Duplicate / key issue (DQ01, DQ02) | Keep every click row; aggregate clicks by enrolment × day (and × week) before any join; build a de-duplicated twin of each click feature (`*_dedup`, exact duplicate rows counted once) | Cause of exact duplicates undecidable: 31.5% equal two-row groups against 24.0% by chance, below chance for quizzes; the grain is the record, not the day; the twin lets Milestone 4 test sensitivity | `step4_keys`: daily aggregation with `records`, `clicks`, `clicks_dedup` | V03 key uniqueness of every view; V04 total clicks = 39,605,099 raw and 38,343,063 de-duplicated |
| Invalid category / type (DQ12, DQ07) | Normalise "10-20" to "10-20%"; ordered categories for imd_band, age_band, highest_education; banked results separated (`status = banked`), never treated as day −1 submissions | Labels must sort and match; day −1 is a placeholder for a carried-over result, not a submission time | `step3_standardise` | V05 every category in its expected set; V06 no banked row in lateness or missed counts |
| Outliers (DQ10, DQ16) | Keep all values: 6,977-click row, 655 credits, submissions up to day 608 | Heavy users, multi-module students and very late submissions are real; capping would change real totals; post-run submissions fall after both cut-offs | No deletion; log1p of counts applied in `fit_transforms.py` for scale-sensitive models | V07 ranges after = before |
| Activity after withdrawal (DQ03) | Truncate engagement measures at the unregistration day in the enrolment, assessment and week views | Activity after leaving is not engagement while enrolled; it would blur AQ1–AQ2 measures | `step6_truncate` | V08 no click or submission counted after the unregistration day |
| Table integration (DQ17, DQ02) | Left joins from the enrolment table; aggregate studentVle and studentAssessment to the target grain first; zero-fill counts | Absence of a row records no activity; inner joins would drop 3,365 enrolments; a naive submissions × clicks join returns 89,347,197 rows | `step7_integrate` with a row-count assertion after every join | V09 rows equal the left table after each join; V10 enrolments without activity kept with 0 |
| Derived features (DQ08, DQ09, DQ18) | Task counts over TMA and CMA only, unweighted; lateness = submission day − due day; missed by the cut-off = due by the cut-off and not submitted by it; score features use CMA marks known by the cut-off (submitted ≤ c − 7), with their count; early-sign rule of C10 (task missed, TMA late or inactive), CMA lateness only in a variant; activity relative to the run's median up to the cut-off, (clicks + 1) ÷ (median + 1); target-derived flags kept out of feature views | Exam scores are missing for 5 modules; weights differ by module; TMA marks have no release day; the rule must fire in every run | `step8_features` (formulas in `feature_spec.md`) | V11 hand-recomputed features for 5 sampled enrolments from raw rows; fixture tests in `package/tests/` |
| Temporal aggregation | Week n = days 7(n−1) to 7n−1 (week 1 = days 0–6, C02); cut-off d13 = end of week 2, d27 = end of week 4; activity before day 0 kept as its own window | The prediction points are fixed in Milestone 2 (C02); pre-start activity is known before teaching and separates outcomes | `step8_features`, `step9_views` (student_week) | V13 no feature uses an event dated after its cut-off |
| Encoding / scaling | Prepared views keep original scales and labels (tree models use them as they are); `fit_transforms.py` builds model-ready matrices for linear models: ordered categories as ordinal codes with a missing indicator, nominal categories one-hot with levels from training runs, counts log1p, median filling and standardisation fitted on training runs (2013) only | Brief section 11: split before any fitted transform; test runs never inform a parameter | `fit_transforms.py` → `metadata/transform_params.json`, `processed/model_ready_d13/d27.parquet` | V12 parameters recomputed with test rows altered give identical values |
| Sensitive attributes (brief 9.1) | Keep gender, age_band, disability, imd_band, region in every view as audit columns; they are not in the primary predictor set; highest_education, prior attempts and credits are predictors | Fairness checks of AQ3–AQ4 need them; using them as predictors is a policy choice left to Milestone 4 as a comparison | `feature_set` role in the dictionary: id, target, predictor, audit, descriptive | V14 every column has a role; no audit column in the primary predictor list |
| Split | Time split by run: training = 2013B and 2013J runs, test = 2014B and 2014J runs; `paired_run` flag for the 9 module/start-month pairs; `student_in_training_runs` flag for test enrolments of students seen in 2013; tuning in Milestone 4 by run-grouped folds inside 2013 | The claim is prediction for a later run (C07); random splits would mix years whose withdrawal rates differ | `step9_views`: `split`, `paired_run`, `student_in_training_runs` | V15 no run in both partitions; flags recomputed independently |


## Transformation log (Task 5): rows in and out of every step

In [3]:
show('transformation_log.md')

# Transformation log
Written by `src/prepare_data.py` at 2026-11-06T15:24:14Z. One row per operation, in the order run; every join is a checked left join (rows in = rows out).

| Step | Operation | Rows in | Rows out | Note |
| :---- | :---- | :---- | :---- | :---- |
| 1 load | courses.csv (SHA-256 pinned, columns as documented) |  | 22 |  |
| 1 load | assessments.csv (SHA-256 pinned, columns as documented) |  | 206 |  |
| 1 load | vle.csv (SHA-256 pinned, columns as documented) |  | 6,364 |  |
| 1 load | studentInfo.csv (SHA-256 pinned, columns as documented) |  | 32,593 |  |
| 1 load | studentRegistration.csv (SHA-256 pinned, columns as documented) |  | 32,593 |  |
| 1 load | studentAssessment.csv (SHA-256 pinned, columns as documented) |  | 173,912 |  |
| 1 load | studentVle.csv (SHA-256 pinned, columns as documented) |  | 10,655,280 |  |
| 2 population | studentInfo + studentRegistration | 32,593 | 32,593 | left join, rows kept |
| 2 population | + courses (run length) | 32,593 | 32,593 | left join, rows kept |
| 2 population | flag enrolled_d13 | 32,593 | 28,069 | enrolled at the day (C09) |
| 2 population | flag enrolled_d14 | 32,593 | 28,061 | enrolled at the day (C09) |
| 2 population | flag enrolled_d27 | 32,593 | 27,535 | enrolled at the day (C09) |
| 3 standardise | imd_band label '10-20' -> '10-20%' | 3,516 | 0 |  |
| 3 standardise | year, start_month, completed, non_completion, split derived | 32,593 | 32,593 |  |
| 3 standardise | tasks = TMA and CMA (exams excluded, DQ08) | 206 | 182 |  |
| 4 keys | studentVle rows -> enrolment x day (records, clicks, clicks_dedup) | 10,655,280 | 1,808,119 | clicks 39,605,099, de-duplicated 38,343,063 |
| 4 keys | studentVle rows -> distinct enrolment x site x day | 10,655,280 | 8,459,320 |  |
| 5 missing | imd_band missing -> category 'Missing' (no imputation) | 1,111 | 1,111 |  |
| 5 missing | flag registration_missing | 32,593 | 45 |  |
| 5 missing | flag withdrawal_day_missing | 32,593 | 93 |  |
| 5 missing | flag unregistered_not_withdrawn | 32,593 | 9 |  |
| 5 missing | flag unregistered_after_run_end | 32,593 | 1 |  |
| 6 truncate | enrolment x day click records up to the unregistration day | 1,808,119 | 1,802,593 | clicks removed 98,745 |
| 6 truncate | enrolment x site x day records up to the unregistration day | 8,459,320 | 8,435,429 |  |
| 7 integrate | enrolment x task grid (TMA, CMA of the run) | 32,593 | 286,898 |  |
| 7 integrate | + submission (if any) | 286,898 | 286,898 | left join, rows kept |
| 7 integrate | TMA/CMA submissions placed in the grid | 168,953 | 168,953 |  |
| 9 views | + banked results | 32,593 | 32,593 | left join, rows kept |
| 9 views | + concurrent runs | 32,593 | 32,593 | left join, rows kept |
| 9 views | + student seen in training runs | 32,593 | 32,593 | left join, rows kept |
| 9 views | + paired run flag | 32,593 | 32,593 | left join, rows kept |
| 9 views | + first missed task week | 32,593 | 32,593 | left join, rows kept |
| 9 views | + tasks while enrolled | 32,593 | 32,593 | left join, rows kept |
| 9 views | + whole-run activity (truncated at withdrawal) | 32,593 | 32,593 | left join, rows kept |
| 9 views | student_week grid + weekly clicks | 1,052,344 | 1,052,344 | left join, rows kept |
| 9 views | + weekly distinct resources | 1,052,344 | 1,052,344 | left join, rows kept |
| 9 views | + logging-gap flag (DQ15) | 1,052,344 | 1,052,344 | left join, rows kept |
| 9 views | student_week rows (weeks -3..run end, cut at withdrawal) |  | 1,052,344 | clicks before week -3 not in this view: 0 |
| 8 features d13 | + activity up to the cut-off | 28,069 | 28,069 | left join, rows kept |
| 8 features d13 | + distinct resources | 28,069 | 28,069 | left join, rows kept |
| 8 features d13 | + tasks due by the cut-off | 28,069 | 28,069 | left join, rows kept |
| 8 features d13 | + CMA marks known by the cut-off | 28,069 | 28,069 | left join, rows kept |
| 8 features d13 | + tasks submitted before their due day (due after the cut-off) | 28,069 | 28,069 | left join, rows kept |
| 8 features d13 | + run median of clicks up to the cut-off (same run, no outcome) | 28,069 | 28,069 | left join, rows kept |
| 8 features d13 | features built; max event day <= cut-off | 28,069 | 28,069 | early_sign_rule 5,673 |
| 8 features d27 | + activity up to the cut-off | 27,535 | 27,535 | left join, rows kept |
| 8 features d27 | + distinct resources | 27,535 | 27,535 | left join, rows kept |
| 8 features d27 | + tasks due by the cut-off | 27,535 | 27,535 | left join, rows kept |
| 8 features d27 | + CMA marks known by the cut-off | 27,535 | 27,535 | left join, rows kept |
| 8 features d27 | + tasks submitted before their due day (due after the cut-off) | 27,535 | 27,535 | left join, rows kept |
| 8 features d27 | + run median of clicks up to the cut-off (same run, no outcome) | 27,535 | 27,535 | left join, rows kept |
| 8 features d27 | features built; max event day <= cut-off | 27,535 | 27,535 | early_sign_rule 7,933 |
| 10 persist | processed/enrolment.parquet |  | 32,593 | 39 columns |
| 10 persist | processed/assessment_due.parquet |  | 286,898 | 14 columns |
| 10 persist | processed/student_week.parquet |  | 1,052,344 | 10 columns |
| 10 persist | processed/features_d13.parquet |  | 28,069 | 50 columns |
| 10 persist | processed/features_d27.parquet |  | 27,535 | 52 columns |
| 10 persist | processed/capacity_by_run.parquet |  | 44 | 9 columns |


## The prepared views

In [4]:
{p.stem: pl.scan_parquet(p).select(pl.len()).collect().item() for p in sorted(PROCESSED.glob('*.parquet'))}

{'assessment_due': 286898,
 'capacity_by_run': 44,
 'enrolment': 32593,
 'features_d13': 28069,
 'features_d27': 27535,
 'model_ready_d13': 28069,
 'model_ready_d27': 27535,
 'student_week': 1052344}

In [5]:
f27 = pl.read_parquet(PROCESSED / 'features_d27.parquet')
f27.head(5)

code_module,code_presentation,id_student,year,split,paired_run,student_in_training_runs,eval_withdrawal_day_unknown,final_result,non_completion,…,clicks_homepage_0c,gender,age_band,disability,imd_band,region,clicks_w1,clicks_w2,clicks_w3,clicks_w4
str,str,i64,i64,str,bool,bool,bool,str,bool,…,i64,str,str,str,str,str,i64,i64,i64,i64
"""AAA""","""2013J""",11391,2013,"""train""",true,false,false,"""Pass""",false,…,35,"""M""","""55<=""","""N""","""90-100%""","""East Anglian Region""",183,20,100,0
"""AAA""","""2013J""",28400,2013,"""train""",true,false,false,"""Pass""",false,…,75,"""F""","""35-55""","""N""","""20-30%""","""Scotland""",175,66,37,57
"""AAA""","""2013J""",31604,2013,"""train""",true,false,false,"""Pass""",false,…,64,"""F""","""35-55""","""N""","""50-60%""","""South East Region""",22,154,108,25
"""AAA""","""2013J""",32885,2013,"""train""",true,false,false,"""Pass""",false,…,50,"""F""","""0-35""","""N""","""50-60%""","""West Midlands Region""",61,116,37,58
"""AAA""","""2013J""",38053,2013,"""train""",true,false,false,"""Pass""",false,…,119,"""M""","""35-55""","""N""","""80-90%""","""Wales""",135,141,135,39


## Split and leakage (Task 6)

In [6]:
f27.group_by('split').agg(pl.len(), pl.col('code_presentation').unique().sort())

split,len,code_presentation
str,u32,list[str]
"""train""",11811,"[""2013B"", ""2013J""]"
"""test""",15724,"[""2014B"", ""2014J""]"


In [7]:
f27.filter(pl.col('split') == 'train').select(pl.col('non_completion').mean().alias('training non-completion rate'))

training non-completion rate
f64
0.444078


In [8]:
tp = json.loads((META / 'transform_params.json').read_text())
tp['fitted_on'], tp['cutoffs']['d27']['nominal_levels']

('split = train (2013B, 2013J runs)',
 {'code_module': ['AAA', 'BBB', 'DDD', 'EEE', 'FFF', 'GGG'],
  'start_month': ['B', 'J']})

## Early-sign rule: firing rate per run (no outcome used)

In [9]:
pl.read_csv(META / 'rule_rates.csv').filter(pl.col('cutoff_day') == 27)

code_module,code_presentation,split,enrolled,early_sign_rule,early_sign_rule_cma,rule_missed,rule_tma_late,inactive_last7,cma_score_coverage,cutoff_day
str,str,str,i64,f64,f64,f64,f64,f64,f64,i64
"""AAA""","""2013J""","""train""",369,0.3496,0.3496,0.0623,0.1409,0.1762,0.0,27
"""AAA""","""2014J""","""test""",348,0.4109,0.4109,0.0805,0.1523,0.204,0.0,27
"""BBB""","""2013B""","""train""",1530,0.3575,0.3575,0.1497,0.0863,0.251,0.0,27
"""BBB""","""2013J""","""train""",1834,0.3855,0.3855,0.1047,0.0665,0.3184,0.0,27
"""BBB""","""2014B""","""test""",1280,0.4078,0.4078,0.0844,0.0961,0.3336,0.0,27
"""BBB""","""2014J""","""test""",1786,0.3326,0.3326,0.0627,0.0381,0.2856,0.0,27
"""CCC""","""2014B""","""test""",1500,0.2873,1.0,0.14,0.0,0.2233,0.8573,27
"""CCC""","""2014J""","""test""",2021,0.2499,0.9896,0.095,0.0,0.2078,0.0,27
"""DDD""","""2013B""","""train""",1143,0.2441,0.9921,0.2196,0.0201,0.0761,0.0,27


## Validation (Task 8)

In [10]:
show('validation_summary.md')

# Validation summary (Task 8)

Written by `src/validate_prepared.py` at 2026-11-06T15:24:17Z. Each check names its expected and observed value.

| ID | Check | Expected | Observed | Status |
| :---- | :---- | :---- | :---- | :---- |
| V01a | enrolment rows and outcome mix equal raw studentInfo | 32,593; [('Distinction', 3024), ('Fail', 7052), ('Pass', 12361), ('Withdrawn', 10156)] | 32,593; [('Distinction', 3024), ('Fail', 7052), ('Pass', 12361), ('Withdrawn', 10156)] | PASS |
| V01b | students | 28,785 | 28,785 | PASS |
| V01c | population sizes equal the profile (C09) | {'d14': 28061, 'd13': 28069, 'd27': 27535} | {'d14': 28061, 'd13': 28069, 'd27': 27535} | PASS |
| V01d | non-completion count per feature view equals the profile | {'d13': 12708, 'd27': 12156} | {'d13': 12708, 'd27': 12156} | PASS |
| V02 | missing values after = before (no imputation in views) | {'imd_band Missing': 1111, 'date_registration': 45, 'TMA/CMA score null (submitted)': 173} | {'imd_band Missing': 1111, 'date_registration': 45, 'TMA/CMA score null (submitted)': 173} | PASS |
| V03 | key unique in every view | 0 duplicates | {'enrolment': 0, 'assessment_due': 0, 'student_week': 0, 'features_d13': 0, 'features_d27': 0, 'capacity_by_run': 0} | PASS |
| V04a | enrolment clicks + clicks after withdrawal = raw clicks | 39,605,099 | 39,506,354 + 98,745 = 39,605,099 | PASS |
| V04b | student_week clicks = enrolment clicks | 39,506,354 | 39,506,354 | PASS |
| V04c | de-duplicated total ≤ raw total | dedup ≤ raw | 38,247,422 ≤ 39,506,354 | PASS |
| V05 | every category in its expected set | no other label | {'imd_band': [], 'age_band': [], 'highest_education': [], 'final_result': [], 'start_month': [], 'split': []} | PASS |
| V06 | banked results carry no lateness and no missed status | 0 | 0 | PASS |
| V07 | ranges after = before (no capping) | score [0, 100]; credits [30, 655] | {'score': [0.0, 100.0], 'studied_credits': [30, 655]} | PASS |
| V08 | no student_week row after the withdrawal week | 0 | 0 | PASS |
| V09 | every logged join kept its rows | 25 joins, rows in = rows out | 25 | PASS |
| V10 | enrolments without any click row kept with 0 clicks | ≥ 3,365 | 3,554 | PASS |
| V11-d13 | 6 features recomputed from raw for all 28,069 rows (independent code) | 0 mismatches | {'clicks_0c': 0, 'clicks_pre': 0, 'active_days_0c': 0, 'sites_0c': 0, 'tasks_missed_c': 0, 'tasks_late_c': 0} | PASS |
| V13-d13 | pipeline clicks differ from an unfiltered recount exactly for students with later clicks; no target-derived or whole-run column in the view | 26,755 rows with later clicks; no forbidden column | 26,755 rows differ; forbidden [] | PASS |
| V11-d27 | 6 features recomputed from raw for all 27,535 rows (independent code) | 0 mismatches | {'clicks_0c': 0, 'clicks_pre': 0, 'active_days_0c': 0, 'sites_0c': 0, 'tasks_missed_c': 0, 'tasks_late_c': 0} | PASS |
| V13-d27 | pipeline clicks differ from an unfiltered recount exactly for students with later clicks; no target-derived or whole-run column in the view | 25,663 rows with later clicks; no forbidden column | 25,663 rows differ; forbidden [] | PASS |
| V12 | transform parameters identical with and without test rows, equal to the stored ones | identical | d13: identical, d27: identical | PASS |
| V14 | every column documented; predictor lists hold no audit, target or descriptive column | none | {'undocumented': [], 'wrong role': []} | PASS |
| V15 | no run in both partitions; 2013 = train; student flag recomputed | 9 train, 13 test runs; 0 flag mismatches | {'train runs': 9, 'test runs': 13, 'test enrolments of students seen in 2013': 1962, 'flag mismatches': 0} | PASS |

## Reports for the analysis plan

- cma_score_coverage_training_rows: {"d13": 0.0138, "d27": 0.0525}
- kpi_sensitivity: {"completion_day14": 0.5474, "enrolled_day14": 28061, "completion_without_138_flagged": 0.5493, "enrolled_without_flagged": 27962}
- rule_rate_overall: {"d13": 0.2021, "d27": 0.2881}
- rule_rate_cma_variant: {"d13": 0.2021, "d27": 0.4123}
- runs_where_rule_exceeds_list_of_5_per_20: {"13": 7, "27": 10}
- runs_where_rule_exceeds_list_of_3_per_20: {"13": 15, "27": 22}
- rule_rate_by_run_range: {"13": [0.0838, 0.4401], "27": [0.1791, 0.5476]}

Rule firing rates per run and cut-off (no outcome used): `rule_rates.csv`. Five training-run rows per cut-off (two with a missed task, two with a late task where the cut-off has one, one at random) recomputed from raw rows: `manual_check_d13.csv`, `manual_check_d27.csv`.


In [11]:
pl.read_csv(META / 'manual_check_d27.csv')

code_module,code_presentation,id_student,clicks_0c,clicks_pre,active_days_0c,sites_0c,tasks_missed_c,tasks_late_c,clicks_0c_raw,clicks_pre_raw,active_days_0c_raw,sites_0c_raw,tasks_missed_c_raw,tasks_late_c_raw
str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""BBB""","""2013B""",435803,0,29,0,0,1,0,0,29,0,0,1,0
"""BBB""","""2013B""",558703,0,0,0,0,1,0,0,0,0,0,1,0
"""BBB""","""2013B""",466973,168,5,10,19,0,1,168,5,10,19,0,1
"""BBB""","""2013J""",593707,301,37,17,44,0,1,301,37,17,44,0,1
"""BBB""","""2013B""",408112,183,31,13,14,0,0,183,31,13,14,0,0


## Data dictionary (excerpt) and lineage

In [12]:
display(Markdown('\n'.join((META / 'data_dictionary.md').read_text().splitlines()[:30])))

# Prepared data dictionary
Written by `src/validate_prepared.py` at 2026-11-06T15:24:17Z from the prepared files and `src/dictionary.py`. Roles: id, split, target, target_derived, predictor, predictor_extended, rule, audit, descriptive. Day fields are offsets from the start of the module run (day 0); week n = days 7(n−1) to 7n−1.

## enrolment.parquet
one row per enrolment (all 32,593); descriptive and diagnostic base. Rows: 32,593; columns: 39.

| Column | Type | Role | Source | Transformation | Sensitivity |
| :---- | :---- | :---- | :---- | :---- | :---- |
| code_module | String | id | studentInfo.code_module | as recorded | none |
| code_presentation | String | id | studentInfo.code_presentation | as recorded | none |
| id_student | Int64 | id | studentInfo.id_student | as recorded (randomised by the provider) | quasi-identifier |
| gender | String | audit | studentInfo.gender | as recorded | sensitive |
| region | String | audit | studentInfo.region | as recorded | sensitive |
| highest_education | String | predictor | studentInfo.highest_education | ordered; ordinal code in model_ready | sensitive |
| imd_band | String | audit | studentInfo.imd_band | '10-20' → '10-20%'; missing → 'Missing' | sensitive |
| age_band | String | audit | studentInfo.age_band | as recorded (ordered 0-35, 35-55, 55<=) | sensitive |
| num_of_prev_attempts | Int64 | predictor | studentInfo.num_of_prev_attempts | as recorded | none |
| studied_credits | Int64 | predictor | studentInfo.studied_credits | as recorded; outliers kept | none |
| disability | String | audit | studentInfo.disability | as recorded (Y/N) | sensitive |
| final_result | String | target | studentInfo.final_result | as recorded | none |
| date_registration | Int64 | descriptive | studentRegistration.date_registration | as recorded | none |
| date_unregistration | Int64 | descriptive | studentRegistration.date_unregistration | as recorded | none |
| enrolled_d13 | Boolean | descriptive | registration days | enrolled_at(13) | none |
| enrolled_d14 | Boolean | descriptive | registration days | enrolled_at(14) | none |
| enrolled_d27 | Boolean | descriptive | registration days | enrolled_at(27) | none |
| year | Int64 | split | code_presentation | first four characters | none |
| start_month | String | predictor | code_presentation | last character: B = February, J = October | none |
| run_length | Int64 | descriptive | courses.module_presentation_length | renamed | none |
| completed | Boolean | target | studentInfo.final_result | Pass or Distinction | none |
| non_completion | Boolean | target | studentInfo.final_result | Fail or Withdrawn | none |